#Install

In [ ]:
!pip install pymupdf
!pip install pytesseract
!pip install -U langchain-community
!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract
!pip install pypdf

In [ ]:
from read_pdf import read_pdf_and_clean
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [ ]:
path = '/content/drive/MyDrive/fix_systematic_review/After_filter_with_match.xlsx'
df = pd.read_excel(path)

In [ ]:
# Set the source directory and collect all PDF files
path = '/content/drive/MyDrive/fix_systematic_review/papers_2'
source_dir = Path(path)
pdf_files = list(source_dir.glob('*.pdf'))

# Create a mapping between cleaned filenames (e.g., derived from DOI or title)
# and the original filename (without the .pdf extension)
cleaned_to_path = {}

for f in pdf_files:
    original_name = f.stem  # Get the filename without .pdf
    cleaned_name = (
        original_name
        .replace("_2F", "/")
        # .replace("_", " ")
        .replace(":", "")
        .replace("_28", "(")
        .replace("_29", ")")
        .replace("_", "\'")
        .lower()
    )
    cleaned_to_path[cleaned_name] = f.stem


In [ ]:
file_texts = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    match = str(row.get("Matching_Filename")).lower()
    name = row.get("Matching_Filename")
    if match in cleaned_to_path:
        file_path = f"{path}/{cleaned_to_path[match]}.pdf"
        # print(filepath)
        docs = read_pdf_and_clean(file_path)
        pages_text = "\n".join([doc.page_content for doc in docs])
        file_texts[name] = pages_text
    else:
        print(match)

In [ ]:
file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/Implementing Best Practice Guidelines under the Best Practice Spotlight Organisation Facilitators and Barriers for Nurses A Delphi Study.pdf'

In [ ]:
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text

# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/36149424.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text

# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/24060660.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text


# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/9791311.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text

# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/17892348.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text


# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/11962183.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text

# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/24830222.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text


# file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/29240372.pdf'
# docs = read_pdf_and_clean(file_path)
# pages_text = "\n".join([doc.page_content for doc in docs])
# file_texts[name] = pages_text


file_path = '/content/drive/MyDrive/fix_systematic_review/papers_2/23641192.pdf'
docs = read_pdf_and_clean(file_path)
pages_text = "\n".join([doc.page_content for doc in docs])
file_texts[name] = pages_text




✅ Text layer found. Using PyPDFLoader...


In [ ]:
len(file_texts.keys() )

185

In [ ]:
import json

# Save
with open("/content/drive/MyDrive/fix_systematic_review/files_full_texts.json", "w", encoding="utf-8") as f:
    json.dump(file_texts, f, ensure_ascii=False, indent=2)

# Load
with open("/content/drive/MyDrive/fix_systematic_review/files_full_texts.json", "r", encoding="utf-8") as f:
    file_texts = json.load(f)


In [ ]:
prompt_new = """
You are a systematic review expert.
Given the following text from a scientific article:

{text}

Rate to what extent the article meets the criteria for "Review / Opinion / Practice Implementation Report" on a scale from 1 to 5, where:
1 = Not this type at all (meets none of the criteria)
2 = Meets few criteria
3 = Meets some criteria, but evidence is mixed or insufficient
4 = Meets most criteria
5 = Fully meets all criteria

Criteria (any one or more may apply):
1. Summarizes previous research (narrative or structured literature review), OR
2. Provides expert opinion or reflection (explicitly stating the author's interpretation, judgment, or viewpoint), OR
3. Describes implementation of a program/policy/intervention in practice **without formal data collection or systematic analysis**, OR
4. Shares experiential insights, lessons learned, or case descriptions without a defined research methodology.
5. Does **NOT** include systematic collection or analysis of new empirical data (e.g., surveys, interviews, statistical tests).

Absolute exclusion rule:
If the article is clearly an **empirical research study** in one of the following categories, it must be classified as:
Final decision: No
Rating: 1
These research types are:
• Qualitative research – Non-numerical data collected systematically (e.g., interviews, focus groups, thematic analysis).
• Mixed methods – Integrates both qualitative and quantitative data and analyses.
• Randomized controlled trial (RCT) – Random allocation, experimental manipulation, numerical outcome data, causal inference.
• Quantitative non-randomized – Numerical assessment of effects without randomization (e.g., quasi-experimental/comparison groups).
• Quantitative descriptive – Numerical data describing trends or characteristics, no manipulation, using descriptive statistics, and no qualitative analysis.

Important clarification:
If the article **only describes a real-world practice implementation** (e.g., a program, policy, intervention, or initiative carried out in practice) **and**:
- Does not involve systematic collection or analysis of new empirical data,
- Focuses on lessons learned, experiential insights, or expert reflection,
then it should **NOT** be excluded under the absolute exclusion rule and should be classified according to the criteria above (i.e., it may be Yes).

Respond in the following format only:
Final decision:<Yes/No>
Rating: X
Quote: "..."
Justification: ...
"""


In [ ]:
df = df[~df['Matching_Filename'].isna()]

In [ ]:
from openai import OpenAI

# Iterate with progress bar
client = OpenAI(api_key="api_keybv")

# df['systematic_full'] = None
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing abstracts"):

    # Skip rows with existing responses
    if pd.notnull(row['systematic_full']):
        continue
    name = row.Matching_Filename

    text = file_texts[name]

    full_prompt = prompt_new.format(text = text)


    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=2048,
        top_p=0.45,
        messages=[
            {"role": "user", "content": full_prompt}
        ]
    )

    # Extract and store response
    content = response.choices[0].message.content
    print(content)
    df.at[index, 'systematic_full'] = content

Processing abstracts: 100%|██████████| 184/184 [00:03<00:00, 53.08it/s]

Final decision: Yes  
Rating: 4  
Quote: "The purpose of this article is to address barriers encountered and strategies used to improve rural research recruitment and participation in the Promoting Heart Health in Rural Women (PHH) study."  
Justification: The article discusses the implementation of recruitment strategies and barriers faced during the Promoting Heart Health study, sharing experiential insights and lessons learned without formal data collection or systematic analysis. It does not present new empirical data but rather reflects on the challenges and solutions encountered in practice, aligning with the criteria for a practice implementation report.


In [ ]:
df.to_excel('/content/drive/MyDrive/fix_systematic_review/After_filter_with_match_full_systematic_2.xlsx')

In [ ]:
import re

# Extract text after "Final decision:"
df["Final_decision"] = (
    df["systematic_full"]
      .fillna("")
      .str.extract(r"Final\s*decision\s*:\s*([^\n\r]+)", flags=re.I)[0]
      .str.strip()
)

# (Optional) Normalize to only Yes/No
df["Final_decision_norm"] = (
    df["Final_decision"]
      .str.extract(r"\b(Yes|No)\b", flags=re.I)[0]
      .str.title()
)

# Show a preview
df[["DOI", "Title", "Final_decision", "Final_decision_norm"]].head()


/tmp/ipython-input-4054490192.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Final_decision"] = (
/tmp/ipython-input-4054490192.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Final_decision_norm"] = (


,DOI,Title,Final_decision,Final_decision_norm
0,10.1016/s0964339702000125,Barriers to research utilisation: the clinical...,Yes,Yes
1,10.1097/01.NND.0000320688.57308.f9,Nursing faculty mentors as facilitators for ev...,Yes,Yes
2,10.1111/j.1365-2702.2007.02040.x,The spectrum of barriers to and facilitators o...,No,No
3,10.1097/00005110-200503000-00005,Overcoming barriers and promoting the use of r...,No,No
5,10.1111/j.1365-2648.2008.04878.x,"Nursing practice, knowledge, attitudes and per...",No,No


In [ ]:
df.Final_decision.value_counts()

,count
Final_decision,
No,134
Yes,50


In [ ]:
df[df['Final_decision'] == "Yes"].sample(20).to_excel('20_sytematic_full_text.xlsx')